# UC3: Feature Engineering - Candidatos de Upselling

Este notebook genera la tabla maestra de candidatos `(user_id, producto_candidato)` aplicando las reglas de elegibilidad de negocio y calculando un `score_propension` base para cada par.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import pyarrow

ROOT_DIR = Path().resolve().parent.parent
BASE_TXN = ROOT_DIR / "Datathon_Hey_2026_dataset_transacciones 1" / "dataset_transacciones"
BASE_CONV = ROOT_DIR / "Datathon_Hey_dataset_conversaciones 1" / "dataset_conversaciones"
OUTPUT_DIR = ROOT_DIR / "outputs"
FEATURES_DIR = OUTPUT_DIR / "features"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print("Cargando clientes, productos y cashback...")
df_clientes = pd.read_csv(BASE_TXN / "hey_clientes.csv", dtype={"user_id": str})
df_productos = pd.read_csv(BASE_TXN / "hey_productos.csv", dtype={"user_id": str})

try:
    df_cashback = pd.read_csv(OUTPUT_DIR / "uc3_cashback_perdido.csv")
except FileNotFoundError:
    print("ADVERTENCIA: No se encontró uc3_cashback_perdido.csv. Generando dummy...")
    df_cashback = pd.DataFrame({"user_id": df_clientes["user_id"].unique(), "cashback_perdido_mes": 0})

print("Cargando conversaciones...")
try:
    pyarrow.unregister_extension_type('pandas.period')
except Exception:
    pass
df_convs = pd.read_parquet(BASE_CONV / "dataset_50k_anonymized.parquet")

Cargando clientes, productos y cashback...
Cargando conversaciones...


## 1. Mapeo de productos vigentes por usuario

In [2]:
# Lista de todos los productos disponibles
PRODUCTOS_CATALOGO = [
    'hey_pro',
    'tarjeta_credito_hey', 
    'inversion_hey', 
    'seguro_vida', 
    'credito_nomina',
    'credito_personal', 
    'cuenta_negocios', 
    'seguro_compras', 
    'tarjeta_credito_negocios',
    'tarjeta_credito_garantizada', 
    'credito_auto'
]

# Productos que cada usuario YA tiene (activos)
prods_activos = df_productos[df_productos['estatus'] == 'activo'].groupby('user_id')['tipo_producto'].apply(set).reset_index()
prods_activos.rename(columns={'tipo_producto': 'productos_actuales'}, inplace=True)

df_clientes = df_clientes.merge(prods_activos, on='user_id', how='left')
df_clientes['productos_actuales'] = df_clientes['productos_actuales'].apply(lambda x: x if isinstance(x, set) else set())

# Agregamos 'hey_pro' a los productos actuales si es_hey_pro es True
df_clientes.loc[df_clientes['es_hey_pro'] == True, 'productos_actuales'] = df_clientes.loc[df_clientes['es_hey_pro'] == True, 'productos_actuales'].apply(lambda x: x | {'hey_pro'})

## 2. Generación de Señal Conversacional y Cruce de Variables

In [3]:
pattern = r"hey\s?pro|beneficio|cash\s?back|seguro|inversion|prestamo|credito"
df_convs["menciona_producto"] = df_convs["input"].fillna("").str.contains(pattern, case=False, regex=True)
usuarios_interesados = df_convs.groupby("user_id")["menciona_producto"].max().reset_index()

df_master = df_clientes.merge(df_cashback[["user_id", "cashback_perdido_mes"]], on="user_id", how="left")
df_master["cashback_perdido_mes"] = df_master["cashback_perdido_mes"].fillna(0)

df_master = df_master.merge(usuarios_interesados, on="user_id", how="left")
df_master["menciona_producto"] = df_master["menciona_producto"].fillna(False).astype(int)

# Imputación de nulos críticos
df_master["satisfaccion_1_10"] = df_master["satisfaccion_1_10"].fillna(7.0)
df_master["ingreso_mensual_mxn"] = df_master["ingreso_mensual_mxn"].fillna(df_master["ingreso_mensual_mxn"].median())
df_master["score_buro"] = df_master["score_buro"].fillna(df_master["score_buro"].median())
df_master["dias_desde_ultimo_login"] = df_master["dias_desde_ultimo_login"].fillna(30)

## 3. Explosión de Candidatos (user_id, producto_candidato)

In [4]:
print("Explotando combinaciones...")
candidatos = []
for p in PRODUCTOS_CATALOGO:
    temp = df_master[['user_id', 'productos_actuales']].copy()
    temp['producto_candidato'] = p
    # Filtramos usuarios que ya tienen el producto
    temp['ya_lo_tiene'] = temp['productos_actuales'].apply(lambda x: p in x)
    candidatos.append(temp[~temp['ya_lo_tiene']])

df_candidatos = pd.concat(candidatos, ignore_index=True)
df_candidatos = df_candidatos.merge(df_master.drop(columns=['productos_actuales']), on='user_id', how='left')

print(f"Total de pares (usuario, producto) potenciales: {len(df_candidatos)}")

Explotando combinaciones...
Total de pares (usuario, producto) potenciales: 137475


## 4. Reglas de Elegibilidad (Filtros Duros)

In [5]:
def es_elegible(row):
    prod = row['producto_candidato']
    if prod == 'credito_nomina':
        return bool(row['nomina_domiciliada'])
    if prod == 'credito_personal':
        return row['score_buro'] >= 650
    if prod == 'cuenta_negocios' or prod == 'tarjeta_credito_negocios':
        return str(row['ocupacion']).lower() in ['empresario', 'independiente']
    if prod == 'tarjeta_credito_garantizada':
        return row['score_buro'] < 600
    if prod == 'tarjeta_credito_hey':
        return row['score_buro'] >= 600
    if prod == 'credito_auto':
        return row['ingreso_mensual_mxn'] >= 15000 and row['score_buro'] >= 650
    # cuenta_debito, hey_pro, seguro_vida, seguro_compras, inversion_hey son para todos
    return True

df_candidatos['elegible'] = df_candidatos.apply(es_elegible, axis=1)
df_elegibles = df_candidatos[df_candidatos['elegible']].copy()
print(f"Pares (usuario, producto) después de filtros de elegibilidad: {len(df_elegibles)}")

Pares (usuario, producto) después de filtros de elegibilidad: 78563


## 5. Scoring de Propensión Base

In [6]:
def calcular_score(row):
    # Score base dinámico dependiendo del producto
    prod = row['producto_candidato']
    base = 50.0
    
    if prod == 'hey_pro':
        pct_cashback = min(row['cashback_perdido_mes'] / 300, 1.0) # Cap en 300 MXN
        score = base + (pct_cashback * 35) + (int(row['nomina_domiciliada']) * 10) + (int(row['menciona_producto']) * 5)
    elif 'credito' in prod or 'tarjeta' in prod:
        pct_buro = min((row['score_buro'] - 500) / 350, 1.0) if row['score_buro'] > 500 else 0
        score = base + (pct_buro * 30) + (int(row['menciona_producto']) * 10) + (row['satisfaccion_1_10'] * 1.5)
    elif prod == 'inversion_hey':
        pct_ingreso = min(row['ingreso_mensual_mxn'] / 50000, 1.0)
        score = base + (pct_ingreso * 30) + (int(row['menciona_producto']) * 15)
    else:
        score = base + (row['satisfaccion_1_10'] * 2) + (int(row['menciona_producto']) * 15)
        
    return min(max(score, 0), 100)

df_elegibles['score_propension'] = df_elegibles.apply(calcular_score, axis=1).round(2)

# Guardamos en la ruta dictada por el FEATURE_ENGINEERING_PLAN.md
output_file = FEATURES_DIR / "feat_uc3_candidates.parquet"
cols_to_keep = ['user_id', 'producto_candidato', 'score_propension', 'cashback_perdido_mes', 
                'ingreso_mensual_mxn', 'score_buro', 'nomina_domiciliada', 'menciona_producto']

df_final = df_elegibles[cols_to_keep]
df_final.to_parquet(output_file, index=False)
print(f"\nArchivo persistido correctamente en: {output_file}")
display(df_final.head(10))


Archivo persistido correctamente en: C:\Users\nanoj\Documents\GitHub\Datathon-2026\outputs\features\feat_uc3_candidates.parquet


,user_id,producto_candidato,score_propension,cashback_perdido_mes,ingreso_mensual_mxn,score_buro,nomina_domiciliada,menciona_producto
0,USR-00004,hey_pro,52.73,23.38,61000,837,False,0
1,USR-00009,hey_pro,60.00,0.00,19500,665,True,0
2,USR-00010,hey_pro,52.93,25.09,23500,609,False,0
3,USR-00011,hey_pro,61.08,9.22,60000,592,True,0
4,USR-00013,hey_pro,60.63,5.37,26000,612,True,0
5,USR-00014,hey_pro,61.73,14.85,39000,575,True,0
6,USR-00015,hey_pro,60.00,0.00,19500,484,True,0
7,USR-00017,hey_pro,57.11,60.92,25000,662,False,0
8,USR-00019,hey_pro,60.00,0.00,15500,376,True,0
9,USR-00023,hey_pro,61.53,13.10,12500,570,True,0
